# Fine-tuning Qwen3 with Hugging Face Trainer

A hands-on notebook adapted from the [Hugging Face Transformers fine-tuning tutorial](https://huggingface.co/docs/transformers/training). This is an independently written, runnable walkthrough following the source's main stages: **tokenization → data collator → model loading → training configuration → Trainer → evaluation and saving**.

**Original tutorial:** https://huggingface.co/docs/transformers/training

**Important:** This is *causal language modeling* (next-token prediction), not a manually labeled question-answer supervised fine-tuning dataset. The original tutorial demonstrates full-model fine-tuning. This notebook additionally supports **LoRA**, enabled by default to reduce GPU-memory requirements. It deliberately uses a small sample by default for experimentation.

Run cells in order in JupyterLab or VS Code with a Python environment. First execution needs internet access to download packages, dataset and model. Training can be slow, especially without a GPU.

## 1. Install the libraries

Run once, then **restart your kernel if prompted**. Installing PyTorch with CUDA support may require following the [PyTorch installer](https://pytorch.org/get-started/locally/) for your operating system. The following installs the remaining dependencies.

In [ ]:
%pip install -U "transformers>=4.57" "datasets>=3.0" "accelerate>=1.0" "peft>=0.15" "huggingface_hub>=0.30" safetensors

## 2. Check the runtime and adjust experiment settings

A small training subset makes the first run manageable. Set `USE_LORA = False` to follow the source's full-parameter fine-tuning route; that option can need significantly more VRAM. `MAX_LENGTH=128` and `MAX_TRAIN_SAMPLES=200` are *learning defaults*, not settings from the original tutorial.

In [1]:
import sys, torch, transformers, datasets
print("Python:", sys.version.split()[0]); print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__); print("Datasets:", datasets.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))

MODEL_NAME = "Qwen/Qwen3-0.6B"
DATASET_NAME = "karthiksagarn/astro_horoscope"
USE_LORA = True
MAX_LENGTH = 128
MAX_TRAIN_SAMPLES = 200
MAX_EVAL_SAMPLES = 40
TRAIN_EPOCHS = 1
OUTPUT_DIR = "./qwen3-horoscope-finetuned"
SEED = 42

Python: 3.10.11
PyTorch: 2.12.0+cu130
Transformers: 4.57.6
Datasets: 4.3.0
CUDA available: True
GPU: NVIDIA GeForce RTX 4090
VRAM (GB): 23.99


## 3. Hugging Face authentication (optional)

You **do not need to log in** to train and save the public model locally. Only authenticate if you later choose to upload the result. Never hard-code a private access token in a notebook.

In [2]:
# Optional: uncomment to authenticate interactively before uploading.
from huggingface_hub import login
login()

## 4. Load and inspect the dataset

The documentation uses the `horoscope` field from `karthiksagarn/astro_horoscope`. Inspect a sample before tokenizing so you can verify what the model will actually learn.

In [3]:
from datasets import load_dataset
raw = load_dataset(DATASET_NAME, split="train")
print("Rows:", len(raw))
print("Columns:", raw.column_names)
assert "horoscope" in raw.column_names, "The dataset schema has changed: expected 'horoscope'."
print("Example text:", str(raw[0]["horoscope"])[:600])

Rows: 21959
Columns: ['sign', 'category', 'date', 'horoscope']
Example text: There's a great day ahead of you, Aries. You'll be blessed with the ability to solve problems, and others will come looking for you today. You'll listen, understand, and express empathy. You'll be wise enough to find solutions to any issues they present to you. After a day like this, you might ask yourself if you shouldn't work as a therapist.


## 5. Select a small, deterministic training sample

For a serious experiment, increase sample sizes and use a reliable validation split. For this tutorial, first shuffle and split, then limit examples to keep startup time low.

In [4]:
raw = raw.shuffle(seed=SEED).train_test_split(test_size=0.1, seed=SEED)
train_raw = raw["train"].select(range(min(MAX_TRAIN_SAMPLES, len(raw["train"]))))
eval_raw = raw["test"].select(range(min(MAX_EVAL_SAMPLES, len(raw["test"]))))
print("Training samples:", len(train_raw), "Evaluation samples:", len(eval_raw))

Training samples: 200 Evaluation samples: 40


## 6. Tokenize the text

A tokenizer converts human-readable text to token IDs. Truncation prevents oversized training examples. The data collator will create causal language-model labels and pad batches dynamically.

In [5]:
from transformers import AutoTokenizer, DataCollatorForLanguageModeling
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def tokenize(batch):
    return tokenizer(batch["horoscope"], truncation=True, max_length=MAX_LENGTH)

train_tokens = train_raw.map(tokenize, batched=True, remove_columns=train_raw.column_names)
eval_tokens = eval_raw.map(tokenize, batched=True, remove_columns=eval_raw.column_names)
# Filter examples too short to provide a meaningful next-token target.
train_tokens = train_tokens.filter(lambda example: len(example["input_ids"]) >= 2)
eval_tokens = eval_tokens.filter(lambda example: len(example["input_ids"]) >= 2)
print(train_tokens[0].keys())
print("First sequence length:", len(train_tokens[0]["input_ids"]))

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/40 [00:00<?, ? examples/s]

Filter:   0%|          | 0/200 [00:00<?, ? examples/s]

Filter:   0%|          | 0/40 [00:00<?, ? examples/s]

dict_keys(['input_ids', 'attention_mask'])
First sequence length: 57


## 7. Construct the causal language-model data collator

`mlm=False` means **no masked-language-model objective**. Instead, the model learns to predict the next token. Padding labels are ignored in loss computation.

In [6]:
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
batch = data_collator([train_tokens[0], train_tokens[1]])
print({k: tuple(v.shape) for k,v in batch.items()})
print("Number of ignored padding labels:", int((batch["labels"] == -100).sum()))

{'input_ids': (2, 77), 'attention_mask': (2, 77), 'labels': (2, 77)}
Number of ignored padding labels: 20


## 8. Load pretrained Qwen3-0.6B

The source loads the model using `AutoModelForCausalLM`. We select floating-point precision based on available hardware instead of assuming `bf16=True`, which is not supported by many older GPUs. On an 8 GB GPU, full training can still run out of memory; the next step enables LoRA by default.

In [7]:
from transformers import AutoModelForCausalLM
if torch.cuda.is_available():
    major, _ = torch.cuda.get_device_capability()
    model_dtype = torch.bfloat16 if major >= 8 else torch.float16
else:
    model_dtype = torch.float32
print("Model dtype:", model_dtype)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=model_dtype)
model.config.use_cache = False  # compatible with gradient checkpointing
print("Loaded model:", MODEL_NAME)

Model dtype: torch.bfloat16


W1121 15:02:41.482000 68600 Lib\site-packages\torch\utils\_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1121 15:02:41.715000 68600 Lib\site-packages\torch\utils\_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Loaded model: Qwen/Qwen3-0.6B


## 9. Optional: LoRA (memory-saving adaptation)

**This is an extension beyond the original tutorial.** LoRA trains small adapter matrices while freezing the pretrained base weights. It typically needs substantially less optimizer memory than full fine-tuning. This notebook uses LoRA by default for the user's 8 GB GPU; disable it to reproduce the tutorial's basic full-model Trainer workflow.

In [8]:
if USE_LORA:
    from peft import LoraConfig, get_peft_model
    lora_config = LoraConfig(
        r=8, lora_alpha=16, lora_dropout=0.05,
        target_modules=["q_proj", "v_proj"],
        bias="none", task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
else:
    print("LoRA disabled: training all model parameters.")

trainable params: 1,146,880 || all params: 597,196,800 || trainable%: 0.1920


## 10. Configure training

The original source illustrates epochs, batch size, gradient accumulation, checkpointing, learning rate, evaluation and checkpoint saving. We adjust defaults to keep this notebook runnable for a first experiment. Learning rates differ deliberately for LoRA adapters and full-model fine-tuning.

In [11]:
from transformers import TrainingArguments, Trainer
cuda = torch.cuda.is_available()
bf16 = cuda and model_dtype == torch.bfloat16
fp16 = cuda and model_dtype == torch.float16
args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=TRAIN_EPOCHS,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},    learning_rate=2e-4 if USE_LORA else 2e-5,
    bf16=bf16,
    fp16=fp16,
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    report_to="none",
    push_to_hub=False,
    seed=SEED,
)
print(args)

TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=True,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=epoch,
eval_use_gather_object=False,
fp16=False,
fp16_

## 11. Construct the Trainer and fine-tune

Training changes model parameters (or LoRA adapters) to reduce the next-token prediction loss. On a CPU, this could take a long time. If you see CUDA out-of-memory, restart the kernel and reduce `MAX_LENGTH` and the sample count, then rerun.

In [12]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tokens,
    eval_dataset=eval_tokens,
    processing_class=tokenizer,
    data_collator=data_collator,
)
train_result = trainer.train()
print(train_result.metrics)

Epoch,Training Loss,Validation Loss
1,3.324800,3.267612


{'train_runtime': 49.3398, 'train_samples_per_second': 4.054, 'train_steps_per_second': 1.013, 'total_flos': 45269885583360.0, 'train_loss': 3.47376708984375, 'epoch': 1.0}


## 12. Evaluate

Loss is a next-token prediction metric, not a direct measure of factual accuracy or instruction-following quality. Perplexity is `exp(eval_loss)` when the evaluation loss is sufficiently small to exponentiate.

In [13]:
import math
metrics = trainer.evaluate()
print(metrics)
if "eval_loss" in metrics:
    print("Perplexity:", math.exp(metrics["eval_loss"]) if metrics["eval_loss"] < 50 else "too large")

{'eval_loss': 3.2676117420196533, 'eval_runtime': 2.007, 'eval_samples_per_second': 19.93, 'eval_steps_per_second': 19.93, 'epoch': 1.0}
Perplexity: 26.24857605440079


## 13. Generate text from the fine-tuned model

The model was trained on horoscope text, so this example prompts for a short continuation in that style. A tiny sample and one epoch are for demonstration only; useful improvements are not guaranteed.

In [17]:
train_raw[0]

{'sign': 'virgo',
 'category': 'career',
 'date': '2020/11/16',
 'horoscope': "You feel chained to a situation or job that you hate. You feel like there is no escape. There is! You have the power to let go and find something new. Don't doubt yourself. There is no point living your life in a way that you don't truly love."}

In [19]:
#prompt = "Today's horoscope for Aries:"
prompt = "I am kind of stucked into pain."
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
model.eval()
with torch.no_grad():
    generated = model.generate(
        **inputs, max_new_tokens=200, do_sample=True,
        temperature=0.8, top_p=0.9,
        pad_token_id=tokenizer.pad_token_id,
    )
print(tokenizer.decode(generated[0], skip_special_tokens=True))

I am kind of stucked into pain. I think it might be because of a bad decision I made in the past. I need to get out of this rut. I want to be a better person. What do I need to do to get out of this? I think I will need to look for a new partner. I need to spend time with my friends and family. I want to be happy and healthy. I need to be more self-reliant. What do I need to do to get out of this? I want to take a break. I don't want to lose a lot of money. I need to spend more time on myself. I want to be a better person. I need to find out if I can change my ways. I want to be happy. I want to be better than I am. I think I am getting too close to a problem that I cannot fix. I need to stop it. I want to be free and independent. I think I need to be more honest with myself.


## 14. Save locally

If using LoRA, `save_pretrained` saves the **adapter**, not the full merged base model. Reload the same base model and attach the adapter using PEFT for future use. Full-model fine-tuning saves the model weights.

In [ ]:
from pathlib import Path

save_dir = Path(OUTPUT_DIR) / "final"
trainer.save_model(str(save_dir))
tokenizer.save_pretrained(str(save_dir))
print("Saved to:", save_dir.resolve())
print("LoRA adapter only:", USE_LORA)

Saved to: C:\Users\meisa\Downloads\qwen3-horoscope-finetuned\final
LoRA adapter only: True


In [23]:
OUTPUT_DIR = "./qwen3-horoscope-finetuned/final"

from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

base_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen3-0.6B",
    device_map="auto"
)

model = PeftModel.from_pretrained(
    base_model,
    OUTPUT_DIR
)

tokenizer = AutoTokenizer.from_pretrained(
    OUTPUT_DIR
)

model.eval()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen3ForCausalLM(
      (model): Qwen3Model(
        (embed_tokens): Embedding(151936, 1024)
        (layers): ModuleList(
          (0-27): 28 x Qwen3DecoderLayer(
            (self_attn): Qwen3Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=1024, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1024, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Linear(in_fea

In [24]:
from peft import PeftModel

print("Model type:", type(model).__name__)
print("LoRA/PEFT active:", isinstance(model, PeftModel))
print("Model parameters:", sum(p.numel() for p in model.parameters()))
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

Model type: PeftModelForCausalLM
LoRA/PEFT active: True
Model parameters: 597196800
Trainable parameters: 0


## 15. Optional: upload to the Hugging Face Hub

**Not executed automatically.** The source uses `trainer.push_to_hub()` after training. Authenticate first and choose an appropriate repository. Confirm that the training data and output can be shared publicly before uploading.

In [ ]:
# OPTIONAL, only after logging in and reviewing visibility/data rights:
# from huggingface_hub import login
# login()
# trainer.push_to_hub("YOUR_USERNAME/qwen3-horoscope-experiment")

## 16. What to study next

- [Trainer documentation](https://huggingface.co/docs/transformers/main_classes/trainer)
- [Parameter-efficient fine-tuning](https://huggingface.co/docs/peft/index)
- [Hugging Face TRL SFTTrainer](https://huggingface.co/docs/trl/sft_trainer)
- [Original fine-tuning tutorial](https://huggingface.co/docs/transformers/training)

**Conceptual distinction:** Training from unlabeled documents by predicting the next token is self-supervised *continued pretraining* (or causal LM fine-tuning). Supervised instruction fine-tuning typically uses prompt/response demonstrations; it requires different data formatting and often a chat template. **This notebook follows the former objective.**